In [ ]:
# from langchain_openai import ChatOpenAI
# from langgraph.graph import START, END, StateGraph, graph
# from langgraph.prebuilt import ToolNode
# from langgraph.graph.message import add_messages
# from langchain_core.messages import HumanMessage, AIMessage
# from typing import TypedDict

In [ ]:
# llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
# system_prompt = (
#     "You are an AI agent that assists content creators in creating videos for different brands.\n"
#     "You can create scripts for videos and suggest voiceovers based on those scripts.\n"
#     "Give answers in a stepwise format and print each step on a different line."
# )

In [ ]:
# class OverallState(TypedDict):
#     a: str

# class TrendsOutput(TypedDict):
#     private_data: str

# def trends(state: OverallState) -> TrendsOutput:
#     output = {"private_data": "private data from trends"}
#     print(f"Entered node `trends`:\n\tInput: {state}.\n\tReturned: {output}")
#     return output

# class HashtagGenInput(TypedDict):
#     private_data: str

# def hashtag_gen(state: HashtagGenInput) -> OverallState:
#     output = {"private_data": "private data from hashtag_gen"}
#     print(f"Entered node `hashtag_gen`:\n\tInput: {state}.\n\tReturned: {output}")
#     return output

# def llm_output(state: OverallState) -> OverallState:
#     output = {"private_data": "private data from llm_output"}
#     print(f"Entered node `llm_output`:\n\tInput: {state}.\n\tReturned: {output}")
#     return output





In [ ]:
# builder = StateGraph(OverallState)
# builder.add_node("trends", trends)
# builder.add_node("hashtag_gen", hashtag_gen)
# builder.add_node("llm_output", llm_output)

# # Add edges using node names (strings)
# builder.add_edge(START, "trends")
# builder.add_edge("trends", "hashtag_gen")
# builder.add_edge("hashtag_gen", "llm_output")
# builder.add_edge("llm_output", END)

# # Compile the graph
# graph = builder.compile()

# # Invoke the graph
# response = graph.invoke({
#     "a": "set at start"
# })

# print(f"Output of graph invocation: {response}")


Entered node `trends`:
	Input: {'a': 'set at start'}.
	Returned: {'private_data': 'private data from trends'}
Entered node `hashtag_gen`:
	Input: {'private_data': 'private data from trends'}.
	Returned: {'private_data': 'private data from hashtag_gen'}
Entered node `llm_output`:
	Input: {'a': 'set at start'}.
	Returned: {'private_data': 'private data from llm_output'}
Output of graph invocation: {'a': 'set at start'}


In [25]:
# from IPython.display import Image, display
# try:
#     display(Image(graph.get_graph().draw_mermaid_png()))
# except Exception:
#     # This requires some extra dependencies and is optional
#     pass

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv
import os
import requests

# Load environment variables
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 1. Define the shared state across nodes
class OverallState(TypedDict):
    topic: str
    llm_output: str
    script: str
    video: str
    voice_over: str
    search_results: str
    hashtags: str        

# 2. Load LLM and Search Tools
llm_model = ChatOpenAI(model="gpt-4o-mini")
tavily_api_key = os.getenv("TAVILY_API_KEY")
search_tool = TavilySearchResults(api_key=tavily_api_key)

# 3. LLM Reasoning Node
prompt_template = PromptTemplate.from_template(
    """You are a video content assistant.
Given a brand-related topic: {topic} and trending hashtags,
suggest a video theme or idea.
Mention the style, tone, and key points in 2-3 lines."""
)

def llm_reasoning_node(state: OverallState) -> OverallState:
    prompt = prompt_template.format(topic=state["topic"])
    response = llm_model.invoke(prompt)
    print(f"[LLM Reasoning]\n{response.content}")
    return {**state, "llm_output": response.content}

# 4. SerpAPI Enrichment Node
def serpapi_enrichment_node(state: OverallState) -> OverallState:
    topic = state["topic"]
    serpapi_key = os.getenv("SERPAPI_API_KEY")

    params = {
        "engine": "google",
        "q": topic + " trends",
        "api_key": serpapi_key
    }

    try:
        response = requests.get("https://serpapi.com/search", params=params)
        data = response.json()

        snippets = []
        if "organic_results" in data:
            for result in data["organic_results"]:
                if "snippet" in result:
                    snippets.append(result["snippet"])

        # Extract top hashtag-like terms
        hashtags = []
        for snippet in snippets:
            words = snippet.lower().split()
            for word in words:
                clean = ''.join(filter(str.isalpha, word))
                if clean and len(clean) > 4:
                    hashtags.append(f"#{clean}")

        hashtags = list(dict.fromkeys(hashtags))[:5]  # Deduplicate and limit
    except Exception as e:
        print(f"[SerpAPI Error] {e}")
        hashtags = []

    print(f"[SerpAPI] Hashtags: {hashtags}") 
    return {**state, "hashtags": ", ".join(hashtags)}

# 5. Script Creation Node
def create_script_node(state: OverallState) -> OverallState:
    script = f"""Script for topic '{state['topic']}':
{state['llm_output']}

Trending Hashtags: {state['hashtags']}
"""
    print(f"[Script Node]\n{script}")
    return {**state, "script": script}

# 6. Video Creation Node
def create_video_node(state: OverallState) -> OverallState:
    prompt = f"""Using the following script, create a short visual plan or concept for a video:

Script:
{state['script']}

Output should describe the video format, pacing, shots, and any visual ideas. Include hashtag ideas if relevant."""
    response = llm_model.invoke(prompt)
    print(f"[Video Node]\n{response.content}")
    return {**state, "video": response.content}

# 7. Voice-Over Generation Node
def create_voice_over_node(state: OverallState) -> OverallState:
    prompt = f"""Convert the following video script into a voice-over narration:

Script:
{state['script']}

The tone should be energetic and engaging. Keep it natural and concise."""
    response = llm_model.invoke(prompt)
    print(f"[Voice-Over Node]\n{response.content}")
    return {**state, "voice_over": response.content}

# 8. Optional Tavily Search Node
def tavily_search_node(state: OverallState) -> OverallState:
    print(f"[Tavily Search] Searching for topic: {state['topic']}")
    results = search_tool.invoke(state["topic"])
    results_text = "\n".join([r["content"] for r in results])
    print(f"[Tavily Results]\n{results_text[:500]}...")
    return {**state, "search_results": results_text}

# 9. Build the StateGraph
builder = StateGraph(OverallState)

# Add nodes
builder.add_node("llm_reasoning", llm_reasoning_node)
builder.add_node("serpapi_enrichment", serpapi_enrichment_node)
builder.add_node("create_script", create_script_node)
builder.add_node("create_video", create_video_node)
builder.add_node("create_voice", create_voice_over_node)

# Define flow
builder.add_edge(START, "llm_reasoning")
builder.add_edge("llm_reasoning", "serpapi_enrichment")
builder.add_edge("serpapi_enrichment", "create_script")
builder.add_edge("create_script", "create_video")
builder.add_edge("create_video", "create_voice")
builder.add_edge("create_voice", END)

# Compile the graph
graph = builder.compile()



In [49]:

graph.invoke({"topic": "sneakers"})


[LLM Reasoning]
**Video Theme: "Sneaker Culture Evolution: From Function to Fashion"**

**Style & Tone:** Energetic and engaging, with a mix of dynamic visuals and playful commentary. Use fast cuts and upbeat music to maintain viewer interest.

**Key Points:** Explore the journey of sneakers from their practical origins to becoming high-fashion statements. Highlight iconic models, collaborations, and the rise of sneaker culture on social media, using trending hashtags like #SneakerHead, #SoleCulture, and #KickGame to connect with the audience.
[SerpAPI] Hashtags: ['#spring', '#sneaker', '#trends', '#youre', '#about']
[Script Node]
Script for topic 'sneakers':
**Video Theme: "Sneaker Culture Evolution: From Function to Fashion"**

**Style & Tone:** Energetic and engaging, with a mix of dynamic visuals and playful commentary. Use fast cuts and upbeat music to maintain viewer interest.

**Key Points:** Explore the journey of sneakers from their practical origins to becoming high-fashion s

{'topic': 'sneakers',
 'llm_output': '**Video Theme: "Sneaker Culture Evolution: From Function to Fashion"**\n\n**Style & Tone:** Energetic and engaging, with a mix of dynamic visuals and playful commentary. Use fast cuts and upbeat music to maintain viewer interest.\n\n**Key Points:** Explore the journey of sneakers from their practical origins to becoming high-fashion statements. Highlight iconic models, collaborations, and the rise of sneaker culture on social media, using trending hashtags like #SneakerHead, #SoleCulture, and #KickGame to connect with the audience.',
 'script': 'Script for topic \'sneakers\':\n**Video Theme: "Sneaker Culture Evolution: From Function to Fashion"**\n\n**Style & Tone:** Energetic and engaging, with a mix of dynamic visuals and playful commentary. Use fast cuts and upbeat music to maintain viewer interest.\n\n**Key Points:** Explore the journey of sneakers from their practical origins to becoming high-fashion statements. Highlight iconic models, collab